In [9]:
import numpy as np
import sys 

将txt文件中存储的边的信息，转化为邻接矩阵，方便后续求解tsp问题

In [10]:
# 无向图
edges = {}          # 字典，存储边信息 (node1, node2): distance 
nodes = set()       # 节点
with open(r'4城市.txt', 'r') as file:
    next(file)             # 跳过第一行的列名
    for line in file:
        line = line.strip()
        parts = line.split()
        node1 = int(parts[0])
        node2 = int(parts[1])
        distance = int(parts[2])
        print(node1, node2, distance)
        
        edges[(node1, node2)] = distance   # 无向图
        edges[(node2, node1)] = distance
        nodes.add(node1)
        nodes.add(node2)

print(f"边: {edges}")
print(f"节点: {nodes}")

1 2 12
1 3 1
1 4 8
2 3 2
2 4 3
3 4 10
边: {(1, 2): 12, (2, 1): 12, (1, 3): 1, (3, 1): 1, (1, 4): 8, (4, 1): 8, (2, 3): 2, (3, 2): 2, (2, 4): 3, (4, 2): 3, (3, 4): 10, (4, 3): 10}
节点: {1, 2, 3, 4}


In [11]:
# 转化为邻接矩阵
min_node = min(nodes)     # 节点中最小索引，便于索引转换到从0开始
node_count = len(nodes)
matrix = np.full((node_count, node_count), np.inf)  # 初始化距离为无穷大
# 节点到自身的距离为0
for i in range(node_count):
    matrix[i, i] = 0
# 根据边信息，填充邻接矩阵
for (node1, node2), distance in edges.items():
    matrix[node1-min_node, node2-min_node] = distance
print(f"邻接矩阵: \n{matrix}")

邻接矩阵: 
[[ 0. 12.  1.  8.]
 [12.  0.  2.  3.]
 [ 1.  2.  0. 10.]
 [ 8.  3. 10.  0.]]


用深度优先遍历算法，求解TSP问题

In [12]:
min_distance = sys.maxsize      # 记录最短路径的距离和路径
best_path = []
visited = [False] * node_count  # 记录城市是否访问，True已访问

def dfs(current, count, distance, path):
    """ 
    深度优先遍历求解旅行商问题

    :param current: 当前节点索引
    :param count: 已访问城市数, 包括当前节点
    :param distance: 路径距离
    :param path: 旅行商具体路径
    """
    # 更新全局最优值，找到最优解
    global min_distance, best_path   # 用global能在函数内修改全局变量的值，否则只能读取全局变量(针对不可变类型生效)
    # 如果已访问所有城市，且与起始城市相连, 即求解出一条TSP问题的可行解
    # 起始城市可选0节点
    if count == node_count and matrix[current][0] != float('inf'):
        if distance + matrix[current][0] < min_distance:
            min_distance =  distance + matrix[current][0] 
            best_path = path + [0]
        return 
    # 深度优先遍历
    # 遍历所有未访问且当前节点能到达的城市
    for i in range(node_count):
        if not visited[i] and matrix[current][i] != float('inf'):
            # python中可变类型如列表、字典等，修改变量内部的元素值，可以直接在函数内操作，不需要global
            visited[i] = True           # 访问节点
            dfs(i, count+1, distance+matrix[current][i], path+[i]) 
            visited[i] = False          # 回溯到未访问该节点时

visited[0] = True 
dfs(0, 1, 0, [0])
print(f"最短距离: {min_distance}")
print(f"最短路径: {[best_path[i] + 1 for i in range(len(best_path))]}")  # 显示编码和习题一致

最短距离: 14.0
最短路径: [1, 3, 2, 4, 1]
